# FAISS index — sanity check

Builds the dense index from the precomputed Jina V3 chunk embeddings and verifies it is wired up correctly **before** we move on to query encoding.

What we're checking:

1. `flatten_chunks` produces a `(N_chunks, 1024)` `float32` matrix with every row normalized to length 1. It also **drops zero-norm chunks** — 640 of the 186,473 chunks are all-zero placeholders the upstream embedder couldn't represent (empty / near-empty text). After filtering we expect 185,833 rows.
2. `chunk_to_doc` is position-aligned with the matrix and has the same length.
3. `build_index` returns a populated `IndexFlatIP` with `ntotal == N_chunks` and `d == 1024`.
4. **Self-search**: using one of the stored vectors as the query returns that same vector at rank 1 with score ≈ 1.0. This is the geometric "is everything plugged in correctly" check — if it fails, normalization or dtype is wrong somewhere.

No model loading here; the encoder for real queries comes in a later step.

In [ ]:
import numpy as np
from ows_retrieval.data import load_doc_embeddings
from ows_retrieval.dense import flatten_chunks, build_index

emb = load_doc_embeddings()
print('documents loaded:', len(emb))

## 1. Flatten chunks

Expect `(185833, 1024) float32` — that's 186,473 raw chunks (confirmed in `inspect_embeddings.ipynb`) minus 640 zero-norm chunks dropped by `flatten_chunks`.

In [ ]:
chunk_vectors, chunk_to_doc = flatten_chunks(emb)

print('chunk_vectors shape:', chunk_vectors.shape)
print('chunk_vectors dtype:', chunk_vectors.dtype)
print('chunk_to_doc shape :', chunk_to_doc.shape, 'dtype:', chunk_to_doc.dtype)
print('lengths agree      :', len(chunk_vectors) == len(chunk_to_doc))

## 2. Row norms — every row must be unit length

If anything other than ≈ 1.0 shows up here, FAISS will return scores ranked by magnitude (noise), not cosine. Tiny deviations like `0.99999994` are just float32 rounding and are fine. NaNs would mean a zero-norm row slipped past the filter.

In [ ]:
norms = np.linalg.norm(chunk_vectors, axis=1)
print('min norm :', norms.min())
print('max norm :', norms.max())
print('mean norm:', norms.mean())
print('all within 1e-3 of 1.0:', np.allclose(norms, 1.0, atol=1e-3))

## 3. Build the index

In [ ]:
index = build_index(chunk_vectors)

print('index.ntotal:', index.ntotal, '(expected', len(chunk_vectors), ')')
print('index.d     :', index.d, '(expected 1024)')
print('match       :', index.ntotal == len(chunk_vectors) and index.d == 1024)

## 4. Self-search

Pick an arbitrary stored row, hand it back to FAISS as the query, and ask for top-5. The top-1 hit must be that exact row with score ≈ 1.0 (a unit vector dotted with itself).

We also print the doc id of the top hit and compare to the expected doc id from `chunk_to_doc` — confirms the parallel-array mapping survives a round-trip.

In [ ]:
probe_idx = 12345  # arbitrary chunk
query = chunk_vectors[probe_idx:probe_idx + 1]  # shape (1, 1024)

scores, idxs = index.search(query, 5)
print('top-5 idxs  :', idxs[0].tolist())
print('top-5 scores:', [round(float(s), 4) for s in scores[0]])
print()
print('top-1 idx matches probe :', idxs[0][0] == probe_idx)
print('top-1 score ~ 1.0       :', abs(float(scores[0][0]) - 1.0) < 1e-3)
print('top-1 doc_id            :', chunk_to_doc[idxs[0][0]])
print('expected doc_id         :', chunk_to_doc[probe_idx])

### Methodology note — the 640 dropped chunks

`flatten_chunks` drops 640 of the 186,473 raw chunks (~0.34%) because their vectors are all zeros. Worth documenting so this decision isn't re-litigated later.

### Why drop them

- A zero vector has cosine similarity 0 with every possible query. It cannot contribute meaningful ranking signal.
- Dividing a zero vector by its zero norm produces NaN, which then poisons every downstream score. We have to do *something* — either drop or special-case. Dropping is simpler and equivalent in retrieval effect.
- The 640 zero chunks are concentrated in 203 documents: 6 documents have *some* zero chunks (those documents survive in the index via their remaining non-zero chunks), and 197 documents have *all* chunks zero (those documents vanish from the dense index entirely).
- We verified by intersecting with `qrels.txt` that **none of the 197 vanishing documents are flagged as relevant to any topic** in this val split. So filtering does not cost us recall on any evaluated query.

### Fairness caveat vs. BM25

BM25 indexes documents by their text and does not see this filter — so those 197 documents are still retrievable by BM25 and could appear in BM25's result lists as (typically low-quality) false positives. Dense cannot return them at all.

- **Recall@k** is unaffected by this asymmetry: recall counts relevant retrievals, and none of the 197 are relevant.
- **nDCG@k** could in principle be very slightly affected: if BM25 ranks one of these in its top-k it wastes a slot a relevant doc could fill, while dense cannot make that mistake.